# Generate a LaTeX Diff Report

This notebook runs the Pub Assist `git-latexdiff-web` workflow. It prepares `old.zip`, `new.zip`, and `config.json`, then can run the Docker worker to create `diff.pdf`.

It supports both common layouts:

- A real LaTeX project folder containing `manuscript.tex` or `main.tex`.
- A folder containing one Overleaf-style zip, such as `old/old.zip` and `new/new.zip`.

## Rules

- `main_tex` must be the same relative path inside the old and new projects or zips.
- Use `bib = "bibtex"` for ordinary `.bib`/natbib workflows.
- Use `bib = "biber"` for biber workflows.
- Use `bib = None` when the projects already include generated `.bbl` files, or when there are no citations.
- Docker Desktop must be running before `run_worker = True`.

In [ ]:
# ---------- User inputs ----------

# Your current repo layout uses old/old.zip and new/new.zip.
# You can also point these directly to zip files or expanded LaTeX folders.
old_project = r"old"
new_project = r"new"

# This file must exist inside both old and new project zips/folders.
main_tex = "manuscript.tex"

# Use "bibtex", "biber", or None.
bib = "bibtex"

# None uses blue additions and red struck-through deletions.
# You can also use a latexdiff style string such as "UNDERLINE", "CFONT", or "BOLD".
style = None

# Extra options forwarded to git-latexdiff/latexdiff, for example "--math-markup=whole".
other_cmdlines = ""

# Keep True to generate diff.pdf when you run all cells.
# Set False only when you want to validate paths/config without running Docker.
run_worker = True
pull_image = False
debug = False
show_worker_log = False

# A fresh subfolder is created inside this folder on every run.
workspace_root = r"latexdiff_runs/notebook_runs"

In [ ]:
import json
import shutil
import zipfile
from datetime import datetime
from pathlib import Path

from python_files.latexdiff_web import (
    format_command,
    prepare_latexdiff_workspace,
    run_latexdiff_worker,
)

In [ ]:
def normalize_zip_names(zip_path):
    with zipfile.ZipFile(zip_path) as archive:
        return sorted(name.replace("\\", "/") for name in archive.namelist() if not name.endswith("/"))


def zip_contains_main(zip_path, main_tex):
    expected = main_tex.replace("\\", "/")
    return expected in set(normalize_zip_names(zip_path))


def folder_contains_main(folder, main_tex):
    return (Path(folder) / main_tex).exists()


def resolve_project_input(project_path, label, main_tex):
    """Resolve a project folder, a zip file, or a folder containing one usable zip."""
    path = Path(project_path)

    if path.is_file() and path.suffix.lower() == ".zip":
        if not zip_contains_main(path, main_tex):
            raise FileNotFoundError(f"{label}: {main_tex!r} was not found inside {path}")
        return path

    if path.is_dir():
        if folder_contains_main(path, main_tex):
            return path

        candidate_zips = sorted(path.glob("*.zip"))
        usable_zips = [zip_path for zip_path in candidate_zips if zip_contains_main(zip_path, main_tex)]

        if len(usable_zips) == 1:
            print(f"{label}: using zip found inside folder: {usable_zips[0]}")
            return usable_zips[0]

        if len(usable_zips) > 1:
            raise ValueError(
                f"{label}: multiple zips in {path} contain {main_tex!r}. "
                "Point old_project/new_project directly to the intended zip."
            )

        available = ", ".join(str(zip_path) for zip_path in candidate_zips) or "no zip files"
        raise FileNotFoundError(
            f"{label}: could not find {main_tex!r} directly under {path}, "
            f"and no zip in that folder contains it. Available zips: {available}"
        )

    raise FileNotFoundError(f"{label}: input path does not exist: {path}")


resolved_old_project = resolve_project_input(old_project, "old_project", main_tex)
resolved_new_project = resolve_project_input(new_project, "new_project", main_tex)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
workspace_dir = Path(workspace_root) / run_id

print("Resolved inputs:")
print(f"old_project -> {resolved_old_project}")
print(f"new_project -> {resolved_new_project}")
print(f"main_tex -> {main_tex}")
print(f"workspace_dir -> {workspace_dir}")

In [ ]:
workspace = prepare_latexdiff_workspace(
    old_project=resolved_old_project,
    new_project=resolved_new_project,
    main_tex=main_tex,
    workspace_dir=workspace_dir,
    bib=bib,
    style=style,
    other_cmdlines=other_cmdlines,
    overwrite=True,
)

workspace_path = Path(workspace["workspace"])

print("Prepared git-latexdiff-web workspace:")
for key in ["workspace", "old_zip", "new_zip", "config_json"]:
    print(f"- {key}: {workspace[key]}")

print("\nconfig.json:")
print(json.dumps(workspace["config"], indent=2))

print("\nDocker command:")
print(format_command(workspace["docker_command"]))

In [ ]:
old_files = normalize_zip_names(workspace_path / "old.zip")
new_files = normalize_zip_names(workspace_path / "new.zip")

expected_main = main_tex.replace("\\", "/")
assert expected_main in old_files, f"old.zip does not contain {expected_main}"
assert expected_main in new_files, f"new.zip does not contain {expected_main}"

print(f"Validated old.zip: found {expected_main} and {len(old_files)} file(s).")
print(f"Validated new.zip: found {expected_main} and {len(new_files)} file(s).")

old_bibs = [name for name in old_files if name.lower().endswith((".bib", ".bbl"))]
new_bibs = [name for name in new_files if name.lower().endswith((".bib", ".bbl"))]
print(f"old bibliography files: {old_bibs if old_bibs else 'none'}")
print(f"new bibliography files: {new_bibs if new_bibs else 'none'}")

In [ ]:
if run_worker:
    result = run_latexdiff_worker(
        workspace_dir=workspace_path,
        debug=debug,
        pull_image=pull_image,
        check=False,
    )

    print(f"Worker return code: {result['returncode']}")

    should_show_log = show_worker_log or result["returncode"] != 0

    if should_show_log and result["stdout"]:
        print("\nstdout:\n")
        print(result["stdout"])
    if should_show_log and result["stderr"]:
        print("\nstderr:\n")
        print(result["stderr"])

    diff_pdf = Path(result["diff_pdf"])
    diff_tex = workspace_path / "git-latexdiff" / "new" / main_tex

    if result["returncode"] != 0:
        raise RuntimeError("Docker worker failed. Read stdout/stderr above for the LaTeX error.")
    if not diff_pdf.exists():
        raise RuntimeError("Docker worker finished without creating diff.pdf. Read stdout/stderr above for the LaTeX error.")

    print("\nLatexdiff completed successfully.")
    print(f"diff.pdf: {diff_pdf}")
    print(f"diffed main tex: {diff_tex}")
    if not show_worker_log:
        print("Set show_worker_log=True if you want to see the full LaTeX log.")
else:
    print("run_worker is False. Set run_worker=True after this validation passes and Docker Desktop is running.")

In [ ]:
expected_outputs = [
    workspace_path / "diff.pdf",
    workspace_path / "git-latexdiff" / "new" / main_tex,
]

for path in expected_outputs:
    print(f"{path}: {'FOUND' if path.exists() else 'missing'}")